# 04_05 The full model, and generating text

*📖 Book: sections 4.6 and 4.7*

**How to use this notebook:** click a cell and press Shift+Enter to run just that cell.
Before each code cell, jot down what you think it will do. Then run it and see if you were right.

Run the cells in order, from the top. **A cell can use what earlier cells made**, so if you skip ahead you'll get `NameError: name '…' is not defined`. If that happens, run the cells above it, or just click Run All.

> **Where this fits**
>
> ```
> Odyssey text → numbers → [model reads] → model learns → model writes → MCP tool
> ```
>
> You are here: **model reads**
>
> Project files this connects to:
>
> - `src/homer_ai/gpt.py`
>
> **This notebook:** One block is not a model. Embeddings, a stack of blocks and an output head make the full GPT; it has far more weights than our `model.py`, and can already write (badly, untrained).
>
> For the big picture, what we're building and why, see the [course map](../00_start_here/00_00_course_map.ipynb).

## 1. What you'll learn

- How the full GPT is assembled: embeddings, N blocks, a final norm, the output head
- How many weights a few configs have, against our 11,840 and GPT-2 small
- How one text flows through it, with every shape printed
- How a simple generate() writes one letter at a time (untrained, so gibberish)
- That this model is `src/homer_ai/gpt.py`, checked against the one built here

## 2. Words you'll meet

| Word | Meaning in plain English | Example |
|---|---|---|
| **output head** | the last layer of the model: it turns each letter's 32 numbers into 137 scores, one per token (not an attention head) | `nn.Linear(32, 137)` |
| **logits** | the raw scores the model gives each possible next token, before softmax turns them into chances | 137 logits for each letter |

## 3. The idea

### ❓ How do the pieces go together?

It's the picture from [04_00](04_00_chapter_overview.ipynb), written as code. The model reads the letters, looks up each one's token row **and** position row and adds them ([02_06](../02_text_to_numbers/02_06_positions.ipynb)), sends them through the blocks, normalises once more, and then the output head makes the scores.

```
IDs (1, 5) → token + position (1, 5, 32) → N blocks (1, 5, 32) → final norm → output head → logits (1, 5, 137)
```

We build the block from 04_04 again here, compactly, so that this notebook runs on its own. The new class is `GPTModel` (the last class in the cell).

👉 **Predict first:** the model gets IDs of shape `(1, 5)`. What's the shape of the answer?

In [ ]:
import torch
from torch import nn

from homer_ai.tokenizer import CharTokenizer
from homer_ai.train_homer import BOOKS_DIR, load_books

tokenizer = CharTokenizer(load_books(BOOKS_DIR)[0])


class MultiHeadAttention(nn.Module):  # from 04_04
    def __init__(self, emb_dim, n_heads, context_length, drop_rate):
        super().__init__()
        self.n_heads, self.head_size = n_heads, emb_dim // n_heads
        self.query = nn.Linear(emb_dim, emb_dim, bias=False)
        self.key = nn.Linear(emb_dim, emb_dim, bias=False)
        self.value = nn.Linear(emb_dim, emb_dim, bias=False)
        self.out_proj = nn.Linear(emb_dim, emb_dim)
        self.dropout = nn.Dropout(drop_rate)
        mask = torch.triu(torch.ones(context_length, context_length), diagonal=1).bool()
        self.register_buffer("mask", mask)

    def forward(self, x):
        batch, length, emb_dim = x.shape

        def split(t):
            return t.view(batch, length, self.n_heads, self.head_size).transpose(1, 2)

        q, k, v = split(self.query(x)), split(self.key(x)), split(self.value(x))
        scores = q @ k.transpose(2, 3) / self.head_size**0.5
        scores = scores.masked_fill(self.mask[:length, :length], float("-inf"))
        weights = self.dropout(torch.softmax(scores, dim=-1))
        mixed = weights @ v
        return self.out_proj(mixed.transpose(1, 2).reshape(batch, length, emb_dim))


class TransformerBlock(nn.Module):  # from 04_04
    def __init__(self, cfg):
        super().__init__()
        e = cfg["emb_dim"]
        heads, length, drop = cfg["n_heads"], cfg["context_length"], cfg["drop_rate"]
        self.attention = MultiHeadAttention(e, heads, length, drop)
        self.feed_forward = nn.Sequential(
            nn.Linear(e, 4 * e), nn.GELU(), nn.Linear(4 * e, e)
        )
        self.norm1, self.norm2 = nn.LayerNorm(e), nn.LayerNorm(e)
        self.dropout = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        x = x + self.dropout(self.attention(self.norm1(x)))
        return x + self.dropout(self.feed_forward(self.norm2(x)))


class GPTModel(nn.Module):  # new: the full model
    def __init__(self, cfg):
        super().__init__()
        self.token_embedding = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.position_embedding = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.dropout = nn.Dropout(cfg["drop_rate"])
        blocks = [TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
        self.blocks = nn.Sequential(*blocks)
        self.final_norm = nn.LayerNorm(cfg["emb_dim"])
        self.output_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, ids):
        positions = torch.arange(ids.shape[1])  # 0, 1, 2, … for each place
        x = self.token_embedding(ids) + self.position_embedding(positions)
        x = self.dropout(x)
        x = self.blocks(x)
        x = self.final_norm(x)
        return self.output_head(x)  # the logits


GPT_CONFIG = {
    "vocab_size": tokenizer.vocab_size, "context_length": 24, "emb_dim": 32,
    "n_heads": 2, "n_layers": 2, "drop_rate": 0.1,
}
torch.manual_seed(0)
model = GPTModel(GPT_CONFIG)
ids = torch.tensor([tokenizer.encode("ἄνδρα")])
print("IDs in:    ", tuple(ids.shape))
print("logits out:", tuple(model(ids).shape))

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

`(1, 5)` in, `(1, 5, 137)` out. That's 137 scores for each of the 5 letters, one score per token, the same output shape that `model.py` makes (apart from the leading 1 for "one text"). `torch.arange(ids.shape[1])` gives `0, 1, 2, 3, 4`, the places to look up in the position table. The names `token_embedding` and `output_head` are ours. `model.py` calls them `embedding` and `head`.

</details>

### ❓ Can I follow the shapes step by step?

Yes. We run the same steps one at a time and print the shape after each one. The model is in `eval()` mode, so dropout is off.

👉 **Predict first:** which steps change the shape, and which keep it?

In [ ]:
model.eval()
with torch.no_grad():
    tokens = model.token_embedding(ids)
    places = model.position_embedding(torch.arange(5))
    x = tokens + places
    print("token rows         ", tuple(tokens.shape))
    print("position rows      ", tuple(places.shape), " (the same for every text)")
    print("token + position   ", tuple(x.shape))
    for i, block in enumerate(model.blocks, start=1):
        x = block(x)
        print(f"after block {i}      ", tuple(x.shape))
    x = model.final_norm(x)
    print("after final norm   ", tuple(x.shape))
    print("after output head  ", tuple(model.output_head(x).shape))

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

Only the last step changes the shape (32 numbers become 137 scores). Every other step keeps `(1, 5, 32)`, and that's why we can add as many blocks as we like.

</details>

## 4. Step by step

Each step is one question. Read it, guess what the code will do, run it, then look at what you got.

### ❓ Step 1: how many weights for a few configs?

The size of the model comes from the config. So let's build four, count the real weights, and put them next to our `model.py` and GPT-2 small. We don't build GPT-2 small here (it's huge). Its size is a number everyone quotes: about **124 million** (vocabulary 50,257, context 1,024, 768 numbers, 12 heads, 12 blocks).

👉 **Predict first:** the first config below has 2 blocks. Will it have about 2 times, 3 times or 10 times the weights of `model.py`?

In [ ]:
def count(module):
    return sum(p.numel() for p in module.parameters())


configs = {
    "2 blocks, 32 wide": {**GPT_CONFIG},
    "4 blocks, 32 wide": {**GPT_CONFIG, "n_layers": 4},
    "4 blocks, 64 wide": {**GPT_CONFIG, "n_layers": 4, "emb_dim": 64, "n_heads": 4},
    "6 blocks, 128 wide": {**GPT_CONFIG, "n_layers": 6, "emb_dim": 128, "n_heads": 4},
}
print(f"{'our small model.py':22} {11_840:>12,}")
for name, cfg in configs.items():
    print(f"{name:22} {count(GPTModel(cfg)):>12,}")
print(f"{'GPT-2 small (known)':22} {'124 million':>12}")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

The 2-block model has 34,816 weights, about 3 times `model.py`. Four blocks, 32 wide: 60,032. Four blocks, 64 wide: 218,368. Six blocks, 128 wide: 1,225,728. And GPT-2 small is about 100 times bigger than even that. The weights grow with the number of blocks (a straight line) and with the width (faster). Doubling the width makes a block's own weights grow about 4 times, but the embeddings and the output head only 2 times, so the whole model grew about 3.6 times here (60,032 to 218,368). (The book counts about 163 million for its GPT-2 small, because it keeps the first table and the output head separate. Real GPT-2 shares one table for both, which gives 124 million. Ours keeps them separate, like the book.)

</details>

### ❓ Where do the weights sit?

Here's the first config, part by part. The bars add up to the total.

In [ ]:
import matplotlib.pyplot as plt

from homer_ai.studio_style import ACCENT, INK, studio_axes, studio_figure

parts = {
    "token embedding": count(model.token_embedding),
    "position embedding": count(model.position_embedding),
    "attention (2 blocks)": sum(count(b.attention) for b in model.blocks),
    "feed-forward (2 blocks)": sum(count(b.feed_forward) for b in model.blocks),
    "layer norms (5)": sum(count(b.norm1) + count(b.norm2) for b in model.blocks)
    + count(model.final_norm),
    "output head": count(model.output_head),
}
print("sum of parts:", f"{sum(parts.values()):,}", " model:", f"{count(model):,}")

fig = studio_figure("Where the weights are", "The 2-block, 32-wide model", (10, 6))
ax = studio_axes(fig, (0.28, 0.1, 0.65, 0.7))
names = list(parts)[::-1]
values = [parts[n] for n in names]
ax.barh(names, values, color=ACCENT)
for i, v in enumerate(values):
    ax.text(v + 250, i, f"{v:,}", va="center", color=INK, family="monospace")
ax.set_xlim(0, 1.18 * max(values))
ax.grid(axis="y", visible=False)
ax.set_xlabel("weights")
plt.show()

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

The parts add up exactly to the model's total. The two feed-forward steps are the biggest piece, and the position table and the norms are tiny.

</details>

### ❓ Step 2: how does it write a letter?

Same steps as in [01_04](../01_foundations/01_04_our_model_as_a_black_box.ipynb): feed in the letters so far, take the scores for the **last** letter, pick the best one (that's called greedy, because no chance is involved), add it to the text, and repeat. Two details. The model can only read `context_length` (24) letters, so we feed in only the last 24. And `dim=-1` in `argmax` means "best of the 137 scores".

The model is untrained, so its weights are random. Expect gibberish. Writing real Greek is what chapter 5 is for.

👉 **Predict first:** will it write Greek words?

In [ ]:
def generate(model, ids, n_new, context_length):
    model.eval()
    for _ in range(n_new):
        recent = ids[:, -context_length:]  # at most the last 24 letters
        with torch.no_grad():
            logits = model(recent)
        last = logits[:, -1, :]  # the scores for the last letter
        best = last.argmax(dim=-1, keepdim=True)  # greedy: the highest score
        ids = torch.cat([ids, best], dim=1)  # add the new letter
    return ids


start = torch.tensor([tokenizer.encode("ἄνδρα")])
written = generate(model, start, 12, GPT_CONFIG["context_length"])
print(repr(tokenizer.decode(written[0].tolist())))

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

The first five letters are our own `ἄνδρα`, and then come 12 letters of untrained gibberish. For a model that has learned nothing, that's the right result: the code works, and the weights are random.

</details>

### ❓ Is this model also in the project, so I can train and use it?

Yes. The classes you just read are in `src/homer_ai/gpt.py`, as clean, documented code: `GPTModel`, `TransformerBlock`, `MultiHeadAttention`, `FeedForward`. There's one difference: the sizes live in a `GPTConfig` (a small frozen record), not in a dictionary. Everything else, including the names of the parts, is the same. But is the file's model really the one we built? Let's check: the same config, the same number of weights, the same names and shapes for every weight, and the same answer when it has the same weights.

👉 **Predict first:** will the two have the same number of weights?

In [ ]:
from homer_ai.gpt import GPTConfig
from homer_ai.gpt import GPTModel as ProjectGPT

project = ProjectGPT(GPTConfig(**GPT_CONFIG))
mine = {name: tuple(w.shape) for name, w in model.state_dict().items()}
theirs = {name: tuple(w.shape) for name, w in project.state_dict().items()}
print("weights, this notebook:", f"{count(model):,}")
print("weights, gpt.py:       ", f"{count(project):,}")
print("same names and shapes? ", mine == theirs)

project.load_state_dict(model.state_dict())  # give it OUR weights
project.eval()
with torch.no_grad():
    print("same logits?           ", torch.equal(model(ids), project(ids)))

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

The same 34,816 weights, the same names and shapes, and the same logits once the weights are copied over. So the model we built here **is** `src/homer_ai/gpt.py`. The rest of the course imports it from there, so the classes don't have to be pasted again.

</details>

### ❓ Is this the size of the GPT that ships with the course?

No, and it's worth seeing the two side by side, because both are called "the GPT". The config in this notebook is a **teaching size**: small enough to print every shape. The ready-made GPT in `models/` uses the settings in `GPT_SETTINGS`, at the top of `src/homer_ai/train_homer.py`. Same classes, bigger numbers.

👉 **Predict first:** the shipped model is 4 times as wide and has twice the blocks. About how many times more weights?

In [ ]:
from homer_ai.train_homer import GPT_SETTINGS

shipped = GPTConfig(vocab_size=tokenizer.vocab_size, **GPT_SETTINGS)
print(f"{'setting':16}{'this notebook':>14}{'shipped':>10}")
for key in ("context_length", "emb_dim", "n_heads", "n_layers", "drop_rate"):
    print(f"{key:16}{GPT_CONFIG[key]:>14}{getattr(shipped, key):>10}")
shipped_weights = count(ProjectGPT(shipped))
print(f"{'weights':16}{count(model):>14,}{shipped_weights:>10,}")
print(f"the shipped model has {shipped_weights / count(model):.0f} times more weights")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

| Setting | This notebook | Shipped |
|---|---|---|
| `context_length` | 24 | 64 |
| `emb_dim` | 32 | 128 |
| `n_heads` | 2 | 4 |
| `n_layers` | 2 | 4 |
| `drop_rate` | 0.1 | 0.3 |
| weights | 34,816 | 835,072 |

The shipped model reads 64 letters at once, not 24, is 4 times as wide and has twice the blocks. That gives it 24 times more weights, which is still small enough to train on a laptop in about 3 minutes. Its dropout is higher (0.3) because a model that big can learn Book 1's 20,000 letters by heart. [05_07](../05_training_on_the_odyssey/05_07_train_the_gpt.ipynb) shows why that matters.

</details>

### ❓ What's left to do?

Training. That's chapter 5's job, with the loss and the learning rate from there, and [05_07](../05_training_on_the_odyssey/05_07_train_the_gpt.ipynb) walks through the GPT's own training loop. A ready-trained copy ships with the repo, and `uv run homer-train --model gpt` trains your own (about 3 minutes). `src/homer_ai/model.py` stays as it is: the small one-head model of chapter 1, your starting point.

## 5. Recap

- The full model is: token + position embeddings, N identical blocks, a final layer norm, and an output head that makes 137 scores per letter.
- Real counts: our `model.py` 11,840; 2 blocks 32 wide 34,816; 6 blocks 128 wide 1,225,728; GPT-2 small about 124 million (quoted).
- An untrained model writes gibberish with `generate()`. Training it is chapter 5.

**You can now:**

- name the parts of a GPT in order, from embeddings to output head
- count a model's weights with `sum(p.numel() for p in model.parameters())`

**Next:** [chapter 5: Training on the Odyssey](../05_training_on_the_odyssey/05_00_chapter_overview.ipynb): making the guesses good.